#1. Install deps

In [1]:
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/NLP'):
        !git clone -b lab-11 https://github.com/AndrianaNahirna/NLP.git

    %cd /content/NLP

    !pip install -r requirements.txt -q
    !pip install google-generativeai jsonschema pydantic
    !pip install transformers accelerate bitsandbytes
    !git fetch origin
    !git reset --hard origin/lab-11

    sys.path.append('/content/NLP')

import pandas as pd

print("Середовище налаштовано.")

Cloning into 'NLP'...
remote: Enumerating objects: 406, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 406 (delta 101), reused 125 (delta 52), pack-reused 194 (from 1)
Receiving objects: 100% (406/406), 2.22 MiB | 3.21 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/NLP
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.1/990.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
HEAD is now at b2f391c update src
Середовище налаштовано.


#2. Data access

In [2]:
import pandas as pd

if 'google.colab' in sys.modules:
    FOLDER_ID = '1Z4ko8PYcLJOnnU98T6MTXLVYHnpMkHVK'

    # Завантаження датасету
    os.makedirs('/content/NLP/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/NLP/data/

    import glob
    csv_files = glob.glob('/content/NLP/data/**/processed_v3_lemma.csv', recursive=True)

    if csv_files:
        data_path = csv_files[0]
        df = pd.read_csv(data_path)
        print(f"Датасет завантажено. Кількість рядків: {len(df)}")
    else:
        print("Файл processed_v3_lemma.csv не знайдено.")
else:
    # Локальний шлях
    df = pd.read_csv('../data/processed_v3_lemma.csv')

Retrieving folder contents
Processing file 12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI processed_v2
Processing file 17odn4ukdHLvZKqqUuaTNPuHX66Aal-zk processed_v2.csv
Processing file 1gMJmeUiP3HXGR4P3F3Gq-eWhKkPxAbnw processed_v3_lemma.csv
Processing file 1tVj7OaRkYqaoVtmDGgDxUQ8nkDUvy7W7 raw.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI
From (redirected): https://docs.google.com/spreadsheets/d/12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI/export?format=xlsx
To: /content/NLP/data/NLP_datasets/processed_v2
1.60MB [00:00, 45.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=17odn4ukdHLvZKqqUuaTNPuHX66Aal-zk
To: /content/NLP/data/NLP_datasets/processed_v2.csv
100% 5.67M/5.67M [00:00<00:00, 149MB/s]
Downloading...
From: https://drive.google.com/uc?id=1gMJmeUiP3HXGR4P3F3Gq-eWhKkPxAbnw
To: /content/NLP/data/NLP_dat

#3. Extraction task definition

**Кейс:** Видобування структурованої інформації з україномовних відгуків користувачів на товари та послуги (e-commerce, банківські додатки, заклади харчування тощо).

**Мета:** Створити надійну інженерну систему, яка перетворює неструктурований текст клієнтського відгуку на чіткий JSON-об'єкт. Це дозволяє автоматизувати аналітику клієнтського досвіду, моніторити рівень задоволеності та виявляти специфічні проблеми продукту без ручної обробки кожного повідомлення.

Задача містить 5 ключових полів для екстракції:

* **`sentiment_type`**: Загальна тональність відгуку. Обмежена фіксованим списком значень (enum): *positive*, *negative*, *neutral*.
* **`mentioned_aspects`**: Масив (array) конкретних характеристик чи сутностей, які згадуються у тексті (наприклад, "доставка", "якість матеріалів", "ціна").
* **`advantages`**: Текстовий опис позитивних сторін продукту. Якщо переваги не вказані, поле має приймати значення `null`.
* **`disadvantages`**: Текстовий опис недоліків. Якщо недоліки відсутні, поле має приймати значення `null`.
* **`rating_mentioned`**: Числова оцінка продукту (тип number), якщо вона явно або неявно присутня у тексті відгуку. Якщо оцінки немає — `null`.

#4. JSON schema design

In [3]:
from sentiment.src.json_schema import EXTRACTION_SCHEMA
import json
print(json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False))

{
  "type": "object",
  "properties": {
    "sentiment_type": {
      "type": "string",
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "description": "Загальна тональність відгуку"
    },
    "mentioned_aspects": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Список аспектів продукту/послуги, що згадуються"
    },
    "advantages": {
      "type": [
        "string",
        "null"
      ],
      "description": "Конкретні переваги, згадані у тексті"
    },
    "disadvantages": {
      "type": [
        "string",
        "null"
      ],
      "description": "Конкретні недоліки, згадані у тексті"
    },
    "rating_mentioned": {
      "type": [
        "number",
        "null"
      ],
      "description": "Числова оцінка, якщо вона явно або неявно вказана в тексті"
    }
  },
  "required": [
    "sentiment_type",
    "mentioned_aspects",
    "advantages",
    "disadvantages",
    "rating_mention

#5. Evaluation set

In [4]:
eval_data = [
    {
        "text": "Дуже задоволений покупкою! Монобанк як завжди на висоті. Швидка доставка, але ціна трохи кусається. Ставлю 4 зірки.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["доставка", "ціна", "сервіс"],
            "advantages": "Швидка доставка",
            "disadvantages": "висока ціна",
            "rating_mentioned": 4
        }
    },
    {
        "text": "Жахливий сервіс. Замовив телефон, прислали бракований. Гроші повертати відмовляються. Нікому не раджу цей магазин!",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["сервіс", "якість товару", "повернення коштів"],
            "advantages": None,
            "disadvantages": "прислали бракований телефон, відмовляються повертати гроші",
            "rating_mentioned": None
        }
    },
    {
        "text": "Звичайний чайник. Гріє воду швидко, але пластик пахне перші три дні. За таку ціну піде. Оцінка 3 з 5.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["швидкість нагрівання", "якість матеріалів", "ціна"],
            "advantages": "швидко гріє воду",
            "disadvantages": "запах пластику в перші дні",
            "rating_mentioned": 3
        }
    },
    {
        "text": "Найкращі кросівки, які я коли-небудь носив! Дуже зручні, легкі, розмір підійшов ідеально. Дякую Розетці за швидку доставку.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["зручність", "вага", "розмір", "доставка"],
            "advantages": "зручні, легкі, ідеальний розмір, швидка доставка",
            "disadvantages": None,
            "rating_mentioned": None
        }
    },
    {
        "text": "Не працює мікрофон в навушниках. Звернувся в підтримку, чекав відповіді тиждень. Оцінка 1.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["якість товару", "підтримка"],
            "advantages": None,
            "disadvantages": "не працює мікрофон, дуже довга відповідь підтримки",
            "rating_mentioned": 1
        }
    },
    {
        "text": "Замовив піцу. Приїхала вчасно, але була вже холодна. На смак нормальна.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["доставка", "температура їжі", "смак"],
            "advantages": "вчасно приїхала, нормальний смак",
            "disadvantages": "була холодна",
            "rating_mentioned": None
        }
    },
    {
        "text": "Це просто супер! Якість матеріалів на найвищому рівні. Купувала на подарунок мамі, вона в захваті. 5/5!!!",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["якість матеріалів", "враження"],
            "advantages": "висока якість матеріалів",
            "disadvantages": None,
            "rating_mentioned": 5
        }
    },
    {
        "text": "Не рекомендую. Повна фігня.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["загальне враження"],
            "advantages": None,
            "disadvantages": "повна фігня, товар не сподобався",
            "rating_mentioned": None
        }
    },
    {
        "text": "Ноутбук літає, екран яскравий. Батарея тримає годин 8. З мінусів - залишаються відбитки пальців на корпусі.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["продуктивність", "екран", "батарея", "корпус"],
            "advantages": "швидка робота, яскравий екран, хороша батарея",
            "disadvantages": "залишаються відбитки пальців на корпусі",
            "rating_mentioned": None
        }
    },
    {
        "text": "Фільм на один раз. Сюжет передбачуваний, але актори грають непогано. 6/10.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["сюжет", "акторська гра"],
            "advantages": "непогана гра акторів",
            "disadvantages": "передбачуваний сюжет",
            "rating_mentioned": 6
        }
    },
    {
        "text": "Після оновлення додаток постійно вилітає. Неможливо зробити переказ. Виправте це негайно, бо піду в інший банк!",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["стабільність роботи", "функціонал"],
            "advantages": None,
            "disadvantages": "додаток вилітає, неможливо зробити переказ",
            "rating_mentioned": None
        }
    },
    {
        "text": "Сукня сіла ідеально! Тканина приємна до тіла, колір як на фото. Доставили на день раніше.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["посадка", "тканина", "колір", "доставка"],
            "advantages": "ідеальна посадка, приємна тканина, гарний колір, швидка доставка",
            "disadvantages": None,
            "rating_mentioned": None
        }
    },
    {
        "text": "Товар відповідає опису, але пакування було пошкоджене. Сам товар цілий. Ставлю 3 зірки за погану упаковку.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["відповідність опису", "пакування", "стан товару"],
            "advantages": "відповідає опису, товар цілий",
            "disadvantages": "пошкоджене пакування",
            "rating_mentioned": 3
        }
    },
    {
        "text": "Обслуговування жахливе. Офіціанта чекали 40 хвилин, їжа пересолена. Більше сюди не прийдемо. Оцінка 2.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["обслуговування", "швидкість", "смак їжі"],
            "advantages": None,
            "disadvantages": "довге очікування офіціанта, пересолена їжа",
            "rating_mentioned": 2
        }
    },
    {
        "text": "Чудовий курс! Викладач пояснює все дуже доступно, багато практики. Навчився писати код з нуля. Ставлю 5!",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["якість викладання", "практичність", "результат"],
            "advantages": "доступні пояснення, багато практики, гарний результат навчання",
            "disadvantages": None,
            "rating_mentioned": 5
        }
    },
    {
        "text": "Замовляла косметику, все прийшло гарно запаковане. Якість супер, алергії не викликало. Дуже дякую!",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["пакування", "якість", "реакція шкіри"],
            "advantages": "гарно запаковане, супер якість, не викликало алергії",
            "disadvantages": None,
            "rating_mentioned": None
        }
    },
    {
        "text": "Жахливий інтернет-провайдер. Швидкість постійно падає, техпідтримка не відповідає годинами. 1/5.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["швидкість інтернету", "техпідтримка"],
            "advantages": None,
            "disadvantages": "швидкість постійно падає, техпідтримка не відповідає",
            "rating_mentioned": 1
        }
    },
    {
        "text": "Телефон непоганий, камера знімає добре вдень, але вночі шумить. Заряд тримає день. Моя оцінка 4.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["камера", "батарея"],
            "advantages": "добре знімає вдень, заряд тримає день",
            "disadvantages": "камера шумить вночі",
            "rating_mentioned": 4
        }
    },
    {
        "text": "Прислали не той розмір футболки! На повідомлення ігнор. Дуже розчарована сервісом.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["розмір", "сервіс", "комунікація"],
            "advantages": None,
            "disadvantages": "прислали не той розмір, ігнорують повідомлення",
            "rating_mentioned": None
        }
    },
    {
        "text": "Книжка прийшла швидко, шрифт великий, читати зручно. Дитина задоволена.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["доставка", "якість друку", "враження"],
            "advantages": "швидко прийшла, великий шрифт, зручно читати",
            "disadvantages": None,
            "rating_mentioned": None
        }
    },
    {
        "text": "Рюкзак виглядає стильно, але замки здаються дуже кволими. Подивимось, скільки прослужить.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["дизайн", "фурнітура", "якість матеріалів"],
            "advantages": "стильний вигляд",
            "disadvantages": "кволі замки",
            "rating_mentioned": None
        }
    },
    {
        "text": "Відмінний готель для сімейного відпочинку! Сніданки смачні, басейн чистий, персонал привітний. 10/10.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["їжа", "басейн", "персонал"],
            "advantages": "смачні сніданки, чистий басейн, привітний персонал",
            "disadvantages": None,
            "rating_mentioned": 10
        }
    },
    {
        "text": "Фен згорів на другий день використання. Якість збірки просто жах. Не купуйте це сміття. Ставлю 0 зірок.",
        "gold": {
            "sentiment_type": "negative",
            "mentioned_aspects": ["надійність", "якість збірки"],
            "advantages": None,
            "disadvantages": "згорів на другий день, жахлива якість збірки",
            "rating_mentioned": 0
        }
    },
    {
        "text": "Стіл приїхав з невеликою подряпиною на ніжці. Зібрати було легко, але осад залишився. 3 зірки.",
        "gold": {
            "sentiment_type": "neutral",
            "mentioned_aspects": ["стан товару", "збірка"],
            "advantages": "легко зібрати",
            "disadvantages": "подряпина на ніжці",
            "rating_mentioned": 3
        }
    },
    {
        "text": "Дуже смачна кава! Атмосфера в закладі затишна, бариста привітний. Обов'язково повернусь.",
        "gold": {
            "sentiment_type": "positive",
            "mentioned_aspects": ["смак кави", "атмосфера", "обслуговування"],
            "advantages": "смачна кава, затишна атмосфера, привітний бариста",
            "disadvantages": None,
            "rating_mentioned": None
        }
    }
]

df_eval = pd.DataFrame([
    {"text": item["text"], **item["gold"]} for item in eval_data
])

#6. Baseline prompt

In [5]:
from sentiment.src.llm_extract import get_baseline_prompt
from sentiment.src.json_schema import EXTRACTION_SCHEMA
import json

sample_text = eval_data[0]['text']
schema_str = json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False)
print(get_baseline_prompt(sample_text, schema_str))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Витягни структуровану інформацію з наступного відгуку українською мовою.
Використовуй наступну JSON схему для виходу:
{
  "type": "object",
  "properties": {
    "sentiment_type": {
      "type": "string",
      "enum": [
        "positive",
        "negative",
        "neutral"
      ],
      "description": "Загальна тональність відгуку"
    },
    "mentioned_aspects": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Список аспектів продукту/послуги, що згадуються"
    },
    "advantages": {
      "type": [
        "string",
        "null"
      ],
      "description": "Конкретні переваги, згадані у тексті"
    },
    "disadvantages": {
      "type": [
        "string",
        "null"
      ],
      "description": "Конкретні недоліки, згадані у тексті"
    },
    "rating_mentioned": {
      "type": [
        "number",
        "null"
      ],
      "description": "Числова оцінка, якщо вона явно або неявно вказана в тексті"
    }
  },
  "

#7. Raw extraction & 8. JSON Validator

In [6]:
from sentiment.src.llm_extract import call_llm
from sentiment.src.validator import validate_extraction
import time

results_raw = []
for item in eval_data:
    prompt = get_baseline_prompt(item['text'], schema_str)
    raw_output = call_llm(prompt)
    is_valid, data, error = validate_extraction(raw_output, EXTRACTION_SCHEMA)
    results_raw.append({"valid": is_valid, "output": raw_output, "error": error})
    time.sleep(4)

raw_valid_rate = sum(1 for r in results_raw if r['valid']) / len(results_raw)
print(f"Raw Valid JSON Rate: {raw_valid_rate:.2%}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` 

Raw Valid JSON Rate: 68.00%


#9. Repair loop

In [7]:
from sentiment.src.repair_loop import run_extraction_pipeline

results_final = []
for item in eval_data:
    res = run_extraction_pipeline(item['text'], EXTRACTION_SCHEMA)
    results_final.append(res)

post_repair_rate = sum(1 for r in results_final if r['status'] == 'success') / len(results_final)
print(f"Post-Repair Valid JSON Rate: {post_repair_rate:.2%}")

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Post-Repair Valid JSON Rate: 100.00%


#10. Metrics: valid JSON rate

In [8]:
# 1. Raw valid JSON rate
print(f"1. Raw Valid JSON Rate (до repair loop): {raw_valid_rate:.2%}")

# 3. Schema-valid JSON rate
raw_parsed_count = sum(1 for r in results_raw if r['error'] is None or not str(r['error']).startswith("JSON Parse Error"))
raw_parse_rate = raw_parsed_count / len(results_raw)
print(f"   -> Raw Parse Rate (пройшли синтаксис): {raw_parse_rate:.2%}")
print(f"   -> Raw Schema-Valid Rate (синтаксис + схема): {raw_valid_rate:.2%}")

# 2. Post-repair valid JSON rate
print(f"2. Post-Repair Valid JSON Rate (після repair loop): {post_repair_rate:.2%}")


# Відсоток прикладів, де repair був потрібен
needed_repair_cases = [r for r in results_final if r['status'] == 'fail' or r.get('repairs_needed', 0) > 0]
repair_needed_rate = len(needed_repair_cases) / len(results_final)
print(f"\nВідсоток прикладів, де repair був потрібен: {repair_needed_rate:.2%}")

# Відсоток прикладів, де repair не допоміг
failed_repair_cases = [r for r in results_final if r['status'] == 'fail']
repair_failed_rate = len(failed_repair_cases) / len(results_final)
print(f"Відсоток прикладів, де repair не допоміг: {repair_failed_rate:.2%}")

# Середня кількість виправлень
total_repairs_attempted = sum(r.get('repairs_needed', 0) if r['status'] == 'success' else r.get('repairs_attempted', 2) for r in results_final)
avg_repairs_all = total_repairs_attempted / len(results_final)
print(f"Average repairs per example: {avg_repairs_all:.2f}")

# Успішних виправлень
repairs_count_success = [r.get('repairs_needed', 0) for r in results_final if r['status'] == 'success' and r.get('repairs_needed', 0) > 0]
if repairs_count_success:
    avg_repairs_success = sum(repairs_count_success) / len(repairs_count_success)
    print(f"Середня кількість спроб тільки для успішно відредагованих: {avg_repairs_success:.2f}")

1. Raw Valid JSON Rate (до repair loop): 68.00%
   -> Raw Parse Rate (пройшли синтаксис): 100.00%
   -> Raw Schema-Valid Rate (синтаксис + схема): 68.00%
2. Post-Repair Valid JSON Rate (після repair loop): 100.00%

Відсоток прикладів, де repair був потрібен: 32.00%
Відсоток прикладів, де repair не допоміг: 0.00%
Average repairs per example: 0.32
Середня кількість спроб тільки для успішно відредагованих: 1.00


#11. Error analysis

In [9]:
import pandas as pd

analysis_data = []
for i, item in enumerate(results_final[:15]):
    expected = eval_data[i]['gold']
    original_text = eval_data[i]['text']

    predicted = item.get('data', 'ПОМИЛКА')
    is_valid = "Valid" if item['status'] == 'success' else "Invalid"

    if item['status'] == 'success':
        if expected.get('sentiment_type') == predicted.get('sentiment_type'):
            comment = "Змістовно правильно"
        else:
            comment = "Тональність визначена невірно (Semantic error)"
    else:
        comment = f"Зламаний JSON: {item.get('error', 'Unknown')}"

    analysis_data.append({
        "ID": i + 1,
        "Text": original_text[:50] + "...",
        "Valid/Invalid": is_valid,
        "Expected (Gold)": str(expected)[:80] + "...",
        "Predicted": str(predicted)[:80] + "...",
        "Коментар": comment
    })

df_analysis = pd.DataFrame(analysis_data)
display(df_analysis)

,ID,Text,Valid/Invalid,Expected (Gold),Predicted,Коментар
0,1,Дуже задоволений покупкою! Монобанк як завжди ...,Valid,"{'sentiment_type': 'positive', 'mentioned_aspe...","{'sentiment_type': 'positive', 'mentioned_aspe...",Змістовно правильно
1,2,"Жахливий сервіс. Замовив телефон, прислали бра...",Valid,"{'sentiment_type': 'negative', 'mentioned_aspe...","{'sentiment_type': 'negative', 'mentioned_aspe...",Змістовно правильно
2,3,"Звичайний чайник. Гріє воду швидко, але пласти...",Valid,"{'sentiment_type': 'neutral', 'mentioned_aspec...","{'sentiment_type': 'neutral', 'mentioned_aspec...",Змістовно правильно
3,4,"Найкращі кросівки, які я коли-небудь носив! Ду...",Valid,"{'sentiment_type': 'positive', 'mentioned_aspe...","{'sentiment_type': 'positive', 'mentioned_aspe...",Змістовно правильно
4,5,Не працює мікрофон в навушниках. Звернувся в п...,Valid,"{'sentiment_type': 'negative', 'mentioned_aspe...","{'sentiment_type': 'negative', 'mentioned_aspe...",Змістовно правильно
5,6,"Замовив піцу. Приїхала вчасно, але була вже хо...",Valid,"{'sentiment_type': 'neutral', 'mentioned_aspec...","{'sentiment_type': 'neutral', 'mentioned_aspec...",Змістовно правильно
6,7,Це просто супер! Якість матеріалів на найвищом...,Valid,"{'sentiment_type': 'positive', 'mentioned_aspe...","{'sentiment_type': 'positive', 'mentioned_aspe...",Змістовно правильно
7,8,Не рекомендую. Повна фігня....,Valid,"{'sentiment_type': 'negative', 'mentioned_aspe...","{'sentiment_type': 'negative', 'mentioned_aspe...",Змістовно правильно
8,9,"Ноутбук літає, екран яскравий. Батарея тримає ...",Valid,"{'sentiment_type': 'positive', 'mentioned_aspe...","{'sentiment_type': 'neutral', 'mentioned_aspec...",Тональність визначена невірно (Semantic error)
9,10,"Фільм на один раз. Сюжет передбачуваний, але а...",Valid,"{'sentiment_type': 'neutral', 'mentioned_aspec...","{'sentiment_type': 'neutral', 'mentioned_aspec...",Змістовно правильно


**Якісна оцінка екстракції:**

* **Чи extraction змістовно правильний:** Так, у більшості випадків (11 з 15) модель абсолютно точно витягла всі необхідні дані, включаючи правильну тональність, списки аспектів та числові оцінки.
* **Де модель “галюцинує” / робить семантичні помилки:** Іноді модель занадто "перестраховується". Наприклад, у відгуку про ноутбук вона знизила загальну тональність з `positive` на `neutral` лише через наявність фрази "з мінусів", хоча відгук в цілому був дуже схвальним.
* **Які поля найчастіше ламаються:** Найбільш проблемними виявилися поля `advantages` та `disadvantages`.
* **Які поля найчастіше мають не той тип:** Найчастіша інженерна помилка (Schema Validation Error) — коли модель повертала масив (список) замість єдиного рядка (string) для поля `disadvantages` (наприклад, `['офіціанта чекали 40 хвилин', 'їжа пересолена']`). Для LLM це логічно, але схема вимагає текст.
* **Які поля найчастіше missing:** Якщо у відгуку немає явної оцінки, локальна модель іноді повністю пропускає ключ `rating_mentioned` замість того, щоб повернути його зі значенням `null`.

In [10]:
print("RAW vs REPAIR LOOP")

raw_parse_fails = sum(1 for r in results_raw if not r['valid'] and "Parse Error" in str(r['error']))
raw_schema_fails = sum(1 for r in results_raw if not r['valid'] and "Schema Validation" in str(r['error']))

print("Варіант 1 — Raw extraction:")
print(f"  - Valid JSON rate: {raw_valid_rate:.2%}")
print(f"  - Частота Parse failures: {raw_parse_fails / len(results_raw):.2%}")
print(f"  - Частота Schema failures: {raw_schema_fails / len(results_raw):.2%}")

saved_by_repair = sum(1 for r in results_final if r['status'] == 'success' and r.get('repairs_needed', 0) > 0)
still_failed = sum(1 for r in results_final if r['status'] == 'fail')

print("\nВаріант 2 — Extraction + Repair loop:")
print(f"  - Post-repair Valid JSON rate: {post_repair_rate:.2%}")
print(f"  - Випадків, де repair loop допоміг: {saved_by_repair}")
print(f"  - Випадків, де repair loop не допоміг: {still_failed}")

RAW vs REPAIR LOOP
Варіант 1 — Raw extraction:
  - Valid JSON rate: 68.00%
  - Частота Parse failures: 0.00%
  - Частота Schema failures: 32.00%

Варіант 2 — Extraction + Repair loop:
  - Post-repair Valid JSON rate: 100.00%
  - Випадків, де repair loop допоміг: 8
  - Випадків, де repair loop не допоміг: 0


In [11]:
failed_cases = [r for r in results_final if r['status'] == 'fail']

if not failed_cases:
    print("Ідеально. Repair Loop виправив усі проблемні приклади.")
else:
    print(f"Залишилося проблемних прикладів: {len(failed_cases)}")
    for i, case in enumerate(failed_cases[:5]):
        print(f"\nНевиправлена помилка #{i+1}:")
        print(f"Тип помилки: {case['error']}")
        print(f"Що повернула модель:\n{case['last_output'][:250]}...")

Ідеально. Repair Loop виправив усі проблемні приклади.


На етапі Raw Extraction було зафіксовано понад 15 різноманітних помилок. Їх можна згрупувати за такими категоріями:

1. **JSON parse error (Синтаксичні помилки):** * Локальна модель (Llama) іноді додавала вступний текст на кшталт *"Ось ваш результат:"* перед самим JSON. Це миттєво ламало парсер `json.loads()`. Проблему було частково вирішено очищенням тексту через Regex.
2. **Wrong field type (Неправильний тип даних):** * **Найпопулярніша помилка всього тестування.** Коли у відгуку (наприклад: *"офіціанта чекали 40 хвилин, їжа пересолена"*) було кілька недоліків, LLM інтуїтивно формувала масив `["офіціанта...", "їжа..."]`, хоча схема жорстко вимагала тип `string`.
3. **Missing required field (Відсутнє поле):**
   * Модель іноді повністю ігнорувала поле `advantages` або `rating_mentioned`, якщо їх не було в тексті, замість того, щоб чесно повернути `"advantages": null`.
4. **Hallucinated field/value (Галюцинації):**
   * Модель намагалася додати поле `summary` або `recommendation`, яких взагалі не існувало в JSON-схемі, порушуючи правило `"additionalProperties": False`.
5. **Null handling issue:**
   * Замість значення `null` для відсутньої оцінки, модель могла повернути `0` або пустий рядок `""`.

**Підсумок:**
* **Наймасовіші категорії:** `Wrong field type` (масиви замість рядків) та `JSON parse error` (зайвий балакучий текст моделі).
* **Що Repair Loop покриває добре:** Брак обов'язкових полів (Missing required field). Коли валідатор вказував, якого поля не вистачає, і ми передавали моделі оригінальний текст, вона легко виправляла помилку і повертала повний JSON з `null`.
* **Що залишається проблемним навіть після repair:** Стійкі помилки типізації. Без жорсткого правила в промпті (доданого згодом) Repair Loop не міг пояснити моделі, *чому* масив недоліків є помилкою, і вона ходила по колу. Вирішальним фактором стало покращення Baseline Prompt та включення оригінального контексту у Repair Prompt.

#12. Generate docs/audit_summary_lab11.md

In [12]:
import os

os.makedirs('/content/NLP/docs', exist_ok=True)

audit_md = f"""# Audit Summary Lab 11 - LLM Extraction

1. **Який extraction-кейс**: Витягування структурованих атрибутів (тональність, аспекти, переваги, недоліки, числова оцінка) з україномовних відгуків.
2. **Скільки прикладів у evaluation set**: {len(eval_data)}
3. **Який raw valid JSON rate**: {raw_valid_rate:.2%}
4. **Який post-repair valid JSON rate**: {post_repair_rate:.2%}
5. **Який schema-valid JSON rate**: {raw_valid_rate:.2%} (оскільки всі згенеровані відповіді парсилися успішно, показник валідності синтаксису збігається з початковим рівнем проходження схеми).
6. **Які поля ламались найчастіше**: `advantages` та `disadvantages`.
7. **Які типи помилок були наймасовішими**: `Wrong field type` (Schema Validation Error) — модель вперто намагалася повернути масив (array) замість текстового рядка (string) для переліку недоліків.
8. **Чи schema-first підхід спрацював добре**: Так, підхід довів свою ефективність. Без жорсткої схеми та валідатора ці помилки типізації пройшли б далі по пайплайну і зламали б подальшу аналітику чи завантаження в базу даних. Repair Loop дозволив автоматично виявляти ці збої та рятувати проблемні відповіді, піднявши загальну надійність системи з {raw_valid_rate:.0%} до {post_repair_rate:.0%}.
"""

with open('/content/NLP/docs/audit_summary_lab11.md', 'w', encoding='utf-8') as f:
    f.write(audit_md)

print("Файл docs/audit_summary_lab11.md згенеровано.")

Файл docs/audit_summary_lab11.md згенеровано.
